In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import datetime
from tqdm import tqdm

In [2]:
def get_html(url):
    payload = {}
    headers = {
        'Cookie': 'PHPSESSID=08mqep4o7c7bjp2k9on5av5nh6'
    }

    response = requests.request("GET", url, headers=headers, data=payload, verify=False)
    html_content = response.text

    return html_content

In [3]:
def get_links_and_dates(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    divs = soup.find_all('div', class_='col-md-4 col-sm-6')

    news_links = []

    for div in divs:
        a = div.find('a')
        link = a.get('href')
        time = div.find('time')
        date = time.get('datetime')
        date = datetime.datetime.strptime(date, '%Y-%m-%d')
        link_date = [link, date]
        news_links.append(link_date)

    return news_links

In [4]:
def get_validated_links(news_links, min_date = datetime.datetime(2025,6,1)):
    validated_links = []
    next_page = True
    for link, date in news_links:
        if date < min_date:
            next_page = False
            break
        else:
            validated_links.append([link, date])

    return validated_links, next_page

In [5]:
def get_next_page(lp_url, next_page_number = 1):
    validated_news_links = []
    lp_url = 'https://www.sindipetrolp.org.br/noticias/index.php?pg='
    url = lp_url + str(next_page_number)
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(lp_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    return validated_news_links

In [6]:
def get_content_news(url):
    html_content = get_html(url)
    soup = BeautifulSoup(html_content, 'html.parser')
    
    title = soup.find('h1').text
    paragraphs = soup.find_all('p')

    return title, paragraphs

In [ ]:
def main():
    next_page_number = 0
    validated_news_links = []

    lp_url = 'https://www.sindipetrolp.org.br/noticias/index.php?pg='
    url = lp_url + str(next_page_number)
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(lp_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    result = []
    for url, date in tqdm(validated_news_links):
        title, paragraphs = get_content_news(url)
        num_paragraph = 1
        for paragraph in paragraphs:
            result.append(
                {
                    'sindicato': 'LP',
                    'url' : url,
                    'titulo' : title,
                    'data': str(validated_news_links[0][1]).split(' ')[0],
                    'paragrafo' : paragraph.text,
                    'num_paragrafo' : num_paragraph
                }
            )
            num_paragraph += 1

    return result

In [10]:
result = main()

df = pd.DataFrame(result)

df

c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.sindipetrolp.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.sindipetrolp.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\ronnyldo.silva\Documents\Workspace\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.sindipetrolp.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-

,sindicato,url,titulo,data,paragrafo,num_paragrafo
0,FNP,https://www.sindipetrolp.org.br/noticias/31396...,"A pedido da Petrobrás, FNP esclarece pontos da...",2025-08-18,"Na sexta-feira (15/08), a Federação Nacional d...",1
1,FNP,https://www.sindipetrolp.org.br/noticias/31396...,"A pedido da Petrobrás, FNP esclarece pontos da...",2025-08-18,"Logo no início da reunião, o RH informou que n...",2
2,FNP,https://www.sindipetrolp.org.br/noticias/31396...,"A pedido da Petrobrás, FNP esclarece pontos da...",2025-08-18,A diretoria da FNP reforçou que espera transpa...,3
3,FNP,https://www.sindipetrolp.org.br/noticias/31396...,"A pedido da Petrobrás, FNP esclarece pontos da...",2025-08-18,"Entre as demandas apresentadas pela FNP, estão...",4
4,FNP,https://www.sindipetrolp.org.br/noticias/31396...,"A pedido da Petrobrás, FNP esclarece pontos da...",2025-08-18,LEIA AQUI A PAUTA COMPLETA DA FNP PARA O ACT 2...,5
...,...,...,...,...,...,...
1202,FNP,https://www.sindipetrolp.org.br/noticias/31273...,"Trabalhadores e trabalhadoras do Edisa, no Lit...",2025-08-18,"Como não poderia ser diferente, a greve do adm...",2
1203,FNP,https://www.sindipetrolp.org.br/noticias/31273...,"Trabalhadores e trabalhadoras do Edisa, no Lit...",2025-08-18,"Embora suspensa, a mobilização continua. Na qu...",3
1204,FNP,https://www.sindipetrolp.org.br/noticias/31273...,"Trabalhadores e trabalhadoras do Edisa, no Lit...",2025-08-18,A solidariedade de classe também teve papel fu...,4
1205,FNP,https://www.sindipetrolp.org.br/noticias/31273...,"Trabalhadores e trabalhadoras do Edisa, no Lit...",2025-08-18,"Já a gestão Magda Chambriard, ao insistir na t...",5
